# Inference pipeline — dual orthogonal views

Uses the `biomae` package (`src/biomae/`). See `notebooks/README.md`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from biomae.pipeline import GammarusPipeline
from biomae.paths import data_path, checkpoint_path

%matplotlib inline


## 1) Configuration (chemins)

Adapte :
- `root`
- `img1`, `img2`
- `yolo_weights`, `mobilenet_meta_json`, `mobilenet_weights_pth`


In [ ]:
img1 = data_path("01_raw/mfi_dataset/val/images/img_A_00049.png")
img2 = data_path("01_raw/mfi_dataset/val/images/img_B_00049.png")

pipeline = GammarusPipeline(
    yolo_weights=checkpoint_path("yolov11n_best.pt"),
    clf_weights=checkpoint_path("mobilenet_v3_v4.pth"),
    clf_meta=checkpoint_path("model_meta.json"),
)


## 2) Charger le modèle de détection (YOLO)


In [ ]:
assert os.path.exists(yolo_weights), f"Poids YOLO introuvables: {yolo_weights}"
det_model = YOLO(yolo_weights)
print("✅ YOLO chargé:", yolo_weights)


# 3) Affichage

In [ ]:

def _bgr_to_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def show_image(path, title=None, figsize=(10, 6)):
    img_bgr = cv2.imread(path)
    assert img_bgr is not None, f"Image illisible: {path}"
    plt.figure(figsize=figsize)
    plt.imshow(_bgr_to_rgb(img_bgr))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

def draw_boxes_bgr(img_bgr, boxes_xyxy, scores=None, labels=None, thickness=2):
    """
    img_bgr: np.ndarray (H,W,3)
    boxes_xyxy: np.ndarray (N,4) en [x1,y1,x2,y2]
    scores: np.ndarray (N,) optionnel
    labels: list[str] optionnel
    """
    out = img_bgr.copy()
    boxes_xyxy = np.asarray(boxes_xyxy)
    if boxes_xyxy.size == 0:
        return out

    for i, (x1, y1, x2, y2) in enumerate(boxes_xyxy):
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), thickness)

        txt = ""
        if labels is not None:
            txt += str(labels[i])
        if scores is not None:
            txt += ("" if txt == "" else " ") + f"{float(scores[i]):.2f}"

        if txt:
            cv2.putText(out, txt, (x1, max(0, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
    return out

def show_image_bgr(img_bgr, title=None, figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(_bgr_to_rgb(img_bgr))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


img_path = img1

results = det_model.predict(source=img_path, imgsz=1024, conf=0.25, iou=0.6, verbose=False)
boxes, scores, cls_ids = yolo_result_to_numpy(results[0])

img_bgr = cv2.imread(img_path)
labels = [str(int(c)) for c in cls_ids]  # ou map vers noms si tu as une liste de classes

annot_bgr = draw_boxes_bgr(img_bgr, boxes, scores=scores, labels=labels)
show_image_bgr(annot_bgr, title="YOLO: boxes (custom)")


conf_filter = 0.25
keep = scores >= conf_filter

annot_bgr = draw_boxes_bgr(
    img_bgr,
    boxes[keep],
    scores=scores[keep],
    labels=[labels[i] for i in np.where(keep)[0]]
)

## 3) Charger le modèle de classification (MobileNetV3)


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

assert os.path.exists(mobilenet_meta_json), f"Meta JSON introuvable: {mobilenet_meta_json}"
assert os.path.exists(mobilenet_weights_pth), f"Poids MobileNet introuvables: {mobilenet_weights_pth}"

with open(mobilenet_meta_json, "r") as f:
    meta_clf = json.load(f)

# Accès correct correspondant à votre script d'entraînement
class_names = meta_clf["labels_map"]["names"]
name_to_idx = {name: i for i, name in enumerate(class_names)}

IDX_C = name_to_idx.get("couple")
IDX_F = name_to_idx.get("femelle")
IDX_M = name_to_idx.get("male")
IDX_I = name_to_idx.get("indeterminee") # Manquait dans votre snippet

NUM_CLASSES = len(class_names)
print(f"Classes chargées ({NUM_CLASSES}): {class_names}")
print(f"Indices: C={IDX_C}, F={IDX_F}, M={IDX_M}, I={IDX_I}")

def build_mobilenetv3(num_classes: int):
    m = models.mobilenet_v3_large(weights=None)
    in_feats = m.classifier[0].in_features
    m.classifier = nn.Sequential(
        nn.Linear(in_feats, 512),
        nn.LayerNorm(512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes),
    )
    return m

clf_model = build_mobilenetv3(NUM_CLASSES).to(device)

state = torch.load(mobilenet_weights_pth, map_location=device)
# Gestion propre du préfixe "module." (DataParallel)
if isinstance(state, dict):
    # Si 'state_dict' est une clé (format standard parfois), on descend d'un niveau
    if "state_dict" in state:
        state = state["state_dict"]
    
    # Nettoyage du préfixe "module."
    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

clf_model.load_state_dict(state, strict=True)
clf_model.eval()
print("✅ MobileNetV3 chargé avec succès.")

## 4) Pré-traitement + décision de classe


In [ ]:
preprocess_512 = transforms.Compose([
    transforms.Resize((CROP_SIZE, CROP_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

USE_ARGMAX = True

# Seuils (utilisés seulement si USE_ARGMAX=False)
thr_c = 0.50
thr_f = 0.40
thr_m = 0.40

def decide_from_probs(probs: torch.Tensor) -> int:
    if USE_ARGMAX:
        return int(torch.argmax(probs).item())

    # Extraction sécurisée des valeurs (0.0 si la classe n'existe pas)
    pc = float(probs[IDX_C]) if IDX_C is not None else 0.0
    pf = float(probs[IDX_F]) if IDX_F is not None else 0.0
    pm = float(probs[IDX_M]) if IDX_M is not None else 0.0
    pi = float(probs[IDX_I]) if IDX_I is not None else 0.0

    # Logique métier hiérarchique
    # 1. Indeterminee : Priorité sécurité (si > seuil, on ne prend pas de risque)
    thr_i = 0.50 # Seuil arbitraire, à ajuster
    if pi >= thr_i:
        return IDX_I

    # 2. Couple
    if pc >= thr_c and pc >= pf and pc >= pm:
        return IDX_C
        
    # 3. Femelle
    if pf >= thr_f and pf >= pc and pf >= pm:
        return IDX_F
        
    # 4. Male (souvent la classe par défaut ou majoritaire)
    if pm >= thr_m and pm >= pc and pm >= pf:
        return IDX_M

    # Fallback classique
    return int(torch.argmax(probs).item())

@torch.no_grad()
def predict_pil(pil_img: Image.Image):
    pil_img = pil_img.convert("RGB")
    x = preprocess_512(pil_img).unsqueeze(0).to(device)  # (1,3,512,512)
    logits = clf_model(x)
    probs = torch.softmax(logits, dim=1)[0].cpu()
    pred_idx = decide_from_probs(probs)
    pred_name = class_names[pred_idx]
    return probs, pred_idx, pred_name


## 5) Affichage


In [ ]:
def show_crops_with_preds(crops_bgr, metas, per_crop_preds):
    if len(crops_bgr) == 0:
        print("Aucun crop à afficher.")
        return

    cols = min(4, len(crops_bgr))
    rows = int(np.ceil(len(crops_bgr) / cols))

    plt.figure(figsize=(3 * cols, 3.5 * rows)) # Un peu plus haut pour le texte
    
    for i, (crop_bgr, m, pred) in enumerate(zip(crops_bgr, metas, per_crop_preds), start=1):
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, i)
        plt.imshow(crop_rgb)
        plt.axis("off")

        # Infos de base
        tag = "COUPLE" if m.get("is_couple", False) else "SINGLE"
        pred_name = pred["pred_name"]
        probs = pred["probs"]
        
        # Construction dynamique du texte des probas pour les 4 classes
        # Ex: "couple: 0.12 | femelle: 0.85..."
        probs_text = []
        for idx, name in enumerate(class_names):
            # On raccourcit les noms pour l'affichage (ex: 'indeterminee' -> 'indet')
            short_name = name[:5] 
            p_val = probs[idx]
            # Met en gras/étoile la classe prédite
            mark = "*" if idx == pred["pred_idx"] else ""
            probs_text.append(f"{mark}{short_name}: {p_val:.2f}")

        # On sépare en 2 lignes si ça fait trop long
        mid = len(probs_text) // 2
        line1 = "  ".join(probs_text[:mid])
        line2 = "  ".join(probs_text[mid:])

        plt.title(
            f"{tag} | PRED: {pred_name.upper()}\n{line1}\n{line2}",
            fontsize=9,
            backgroundcolor='white'
        )

    plt.tight_layout()
    plt.show()


## 6) Pipeline complet : `process_image(...)`

**Oui, `process_image` fait tout** : détection → crops → classification → score final.

Elle te renvoie un dictionnaire :
- `pred_name`, `score` : la sortie finale (par image)
- `n_crops` : nombre de crops utilisés
- `per_crop` : détail par crop (debug/qualité)


In [ ]:
def aggregate_probs(per_crop_probs: list[np.ndarray], mode: str = "mean") -> np.ndarray:
    stack = np.stack(per_crop_probs, axis=0)  # (N,C)
    if mode == "mean":
        return stack.mean(axis=0)
    if mode == "max":
        return stack.max(axis=0)
    raise ValueError(f"Mode d'agrégation inconnu: {mode}")

def process_image(
    img_path: str,
    *,
    yolo_imgsz: int = 1024,
    yolo_conf: float = 0.25,
    yolo_iou: float = 0.60,
    crop_conf_filter: float = 0.25,
    group_iou_thr: float = 0.25,
    agg_mode: str = "mean",
    show_debug: bool = False,
):
    assert os.path.exists(img_path), f"Image introuvable: {img_path}"
    img_bgr = cv2.imread(img_path)
    assert img_bgr is not None, f"Image illisible: {img_path}"

    results = det_model.predict(
        source=img_path,
        imgsz=yolo_imgsz,
        conf=yolo_conf,
        iou=yolo_iou,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
        save=False
    )

    boxes, scores, cls_ids = yolo_result_to_numpy(results[0])
    
    if show_debug:
        labels = [str(int(c)) for c in cls_ids]
        annot_bgr = draw_boxes_bgr(img_bgr, boxes, scores=scores, labels=labels)
        show_image_bgr(annot_bgr, title="Détection YOLO (boxes + scores)")

    crops_bgr, metas = boxes_to_crops_512(
        img_bgr=img_bgr,
        boxes_xyxy=boxes,
        scores=scores,
        conf_filter=crop_conf_filter,
        group_mode="iou",
        group_iou_thr=group_iou_thr,
        canvas_size=CROP_SIZE,
        downscale_only=True
    )

    if len(crops_bgr) == 0:
        return {
            "img": img_path,
            "pred_name": "none",
            "score": 0.0,
            "n_crops": 0,
            "per_crop": [],
        }

    per_crop = []
    per_crop_probs = []

    for crop_bgr, m in zip(crops_bgr, metas):
        pil_crop = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        probs_t, pred_idx, pred_name = predict_pil(pil_crop)
        probs = probs_t.numpy().astype(float)

        per_crop.append({
            "meta": m,
            "pred_idx": int(pred_idx),
            "pred_name": pred_name,
            "probs": probs,
            "score": float(probs[pred_idx]),
        })
        per_crop_probs.append(probs)

    agg_probs = aggregate_probs(per_crop_probs, mode=agg_mode)
    final_idx = int(np.argmax(agg_probs))
    final_name = class_names[final_idx]
    final_score = float(agg_probs[final_idx])

    out = {
        "img": img_path,
        "pred_name": final_name,
        "score": final_score,
        "n_crops": len(crops_bgr),
        "agg_probs": agg_probs,
        "per_crop": per_crop,
    }

    if show_debug:
        show_crops_with_preds(crops_bgr, metas, per_crop)

    return out

## 7) Exécution sur 2 images (séquentiel)


In [ ]:
results = pipeline.process_dual_view(img1, img2)
for label, res in [("View A", results["view_a"]), ("View B", results["view_b"])]:
    print(f"----- {label} -----")
    print("Image:", res["img"])
    print("Pred :", res["pred_name"])
    print(f"Score: {res['score']:.3f} | n_crops: {res['n_crops']}")
